Cron Job
- Triggers the ballot run for a sales exercise
- Fire-and-forget only

ExecuteBallotService
- Orchestrates the whole ballot process
- Calls BallotAuditService to create/update ballot run records
- Calls ProjectService to get projects under the exercise
- Calls FlatService to get available flat counts by flat type
- Calls ApplicationService to get ballot candidate applications
- Calls DocumentService to get extracted fields / document facts
- Calls EligibilityService to decide whether each application can proceed to ballot
- Calls PastApplicationService to get extra chances for qualifying Standard applications
- Runs the ballot logic itself:
  - scheme logic
  - base chance logic
  - extra chance merge
  - weighted ballot
  - queue number generation
- Sends successful results to FlatSelectionService
- Sends notifications through NotificationService

BallotAuditService
- Stores ballot run and bucket run status
- Stores whether the run/bucket succeeded or failed
- Stores error message if failed

ProjectService
- Returns all projects under an exercise
- Each project includes town

FlatService
- Returns available flat counts grouped by flat type for a set of projectIds

ApplicationService
- Returns candidate applications for a bucket
- Returns Application + ApplicationMember data

DocumentService
- Stores documents and extracted fields
- Returns only the fields needed for eligibility checking

EligibilityService
- Decides whether an application is still eligible to proceed into BTO ballot
- Does NOT decide scheme logic
- Does NOT decide ballot chances

PastApplicationService
- Looks at past applications
- Returns only extra chances from past unsuccessful Standard BTO attempts
- Penalty reset logic is handled here

FlatSelectionService
- Receives successful ballot results with queue numbers
- Handles unit booking later

NotificationService
- Sends ballot / booking notifications

Application
- application_id
- exercise_id
- project_id
- flat_type
- priority_scheme


FlatSelection
- FlatSelec
- status
- queue_no
- selected_flat_id

ApplicationMember
- application_member_id
- application_id
- nric
- full_name
- member_role
- contact_no
- email

BallotRun
- ballot_run_id
- exercise_id
- run_at
- status

BallotRunBucket
- bucket_id
- ballot_run_id
- flat_type
- status
- error_message

PastApplication
- past_application_id
- application_id
- main_applicant_nric
- co_applicant_nric
- is_standard
- result_status (SUCCESSFUL, UNSUCCESSFUL)
- penalty_until
- applied_at

Flat
- flat_id
- project_id
- flat_type
- status

Project
- project_id
- exercise_id
- town
- classification

In [ ]:
app ='test'
# Ballot Audits
@app.route('/ballot-audits', methods=['GET'])
@app.route('/ballot-audits', methods=['POST'])
@app.route('/ballot-audits/<int:audit_id>', methods=['GET'])
@app.route('/ballot-audits/<int:audit_id>/status', methods=['PUT'])

# Ballot Runs / Orchestrator
@app.route('/ballot-runs', methods=['GET'])
@app.route('/ballot-runs', methods=['POST'])
@app.route('/ballot-runs/<int:ballot_run_id>', methods=['GET'])
@app.route('/ballot-runs/<int:ballot_run_id>/status', methods=['PUT'])
@app.route('/ballot-runs/<int:ballot_run_id>/results', methods=['GET'])

# Projects
@app.route('/projects', methods=['GET'])
@app.route('/projects/<int:project_id>', methods=['GET'])
@app.route('/exercises/<int:exercise_id>/projects', methods=['GET'])

# Flats
@app.route('/flats', methods=['GET'])
@app.route('/flats/<int:flat_id>', methods=['GET'])
@app.route('/projects/<int:project_id>/flats', methods=['GET'])
@app.route('/flats/availability', methods=['GET'])

# Applications
@app.route('/applications', methods=['GET'])
@app.route('/applications/<int:application_id>', methods=['GET'])
@app.route('/applications/eligible-for-ballot', methods=['GET'])

# Documents / Document Service Wrapper
@app.route('/applications/<int:application_id>/documents', methods=['GET'])
@app.route('/documents/submitted-data', methods=['POST'])

# Eligibility
@app.route('/eligibility-checks', methods=['POST'])
@app.route('/eligibility-checks/<int:check_id>', methods=['GET'])

# HFE Applications
@app.route('/hfe-applications', methods=['GET'])
@app.route('/hfe-applications/<int:hfe_application_id>', methods=['GET'])
@app.route('/hfe-checks', methods=['POST'])
@app.route('/hfe-checks/<int:check_id>', methods=['GET'])

# Past Applications
@app.route('/past-applications', methods=['GET'])
@app.route('/applicants/<int:applicant_id>/past-applications', methods=['GET'])
@app.route('/past-application-ballot-chances', methods=['POST'])
@app.route('/past-application-outcomes', methods=['PUT'])

# Ballot Engine
@app.route('/ballots', methods=['POST'])
@app.route('/ballots/<int:ballot_id>', methods=['GET'])
@app.route('/ballots/<int:ballot_id>/results', methods=['GET'])

# Flat Selections
@app.route('/flat-selections', methods=['GET'])
@app.route('/flat-selections', methods=['POST'])
@app.route('/flat-selections/<int:selection_id>', methods=['GET'])
@app.route('/flat-selections/<int:selection_id>/status', methods=['PUT'])
@app.route('/flat-selections/bulk', methods=['POST'])

# Applicants
@app.route('/applicants', methods=['GET'])
@app.route('/applicants/<int:applicant_id>', methods=['GET'])
@app.route('/applicants/<int:applicant_id>/contact-details', methods=['GET'])
@app.route('/applicant-contact-details', methods=['POST'])

# Notifications
@app.route('/notifications', methods=['POST'])
@app.route('/notifications/<int:notification_id>', methods=['GET'])
@app.route('/ballot-result-notifications', methods=['POST'])

# Errors
@app.route('/errors', methods=['POST'])
@app.route('/errors/<int:error_id>', methods=['GET'])

### Step 1 — Cron starts ballot run
`POST /execute-ballot/run`

```json
{
  "exerciseId": 1,
  "runAt": "2026-03-21T10:00:00Z"
}

### Step 2 — Execute Ballot creates ballot run audit record

`POST /ballot-audit/runs`

```json
{
  "exerciseId": 1,
  "runAt": "2026-03-21T10:00:00Z",
  "status": "STARTED"
}

### Step 3 — Execute Ballot gets projects under the exercise

`GET /projects/exercise/1`

Response
```json
[
  {
    "projectId": 11,
    "exerciseId": 1,
    "town": "Woodlands"
  },
  {
    "projectId": 12,
    "exerciseId": 1,
    "town": "Woodlands"
  },
  {
    "projectId": 13,
    "exerciseId": 1,
    "town": "Yishun"
  }
]

### Step 4 — Execute Ballot gets available flat counts for one project group

`POST /flats/available-counts`

```json
{
  "projectIds": [11, 12]
}

Response
{
  "counts": [
    {
      "flatType": "2-Room Flexi",
      "availableCount": 40
    },
    {
      "flatType": "3-Room",
      "availableCount": 120
    },
    {
      "flatType": "4-Room",
      "availableCount": 280
    }
  ]
}

### Step 5 — Execute Ballot gets ballot candidate applications for one bucket
`POST /applications/ballot-candidates` 
either this or `GET /applications/<exerciseId>`
THEN RETURN ALL ELIGIBLE APPLICATIONS

```json
{
  "exerciseId": 1,
  "projectIds": [11, 12],
  "flatType": "4-Room"
}

Response
[
  {
    "applicationId": 101,
    "flatType": "4-Room",
    "ballotScheme": "FPPS",
    "members": [
      {
        "applicationMemberId": 1,
        "nric": "S1234567A",
        "memberRole": "MAIN_APPLICANT"
      },
      {
        "applicationMemberId": 2,
        "nric": "S2345678B",
        "memberRole": "CO_APPLICANT"
      }
    ]
  },
  {
    "applicationId": 102,
    "flatType": "4-Room",
    "ballotScheme": null,
    "members": [
      {
        "applicationMemberId": 3,
        "nric": "S3456789C",
        "memberRole": "MAIN_APPLICANT"
      }
    ]
  }
]

### Step 6 — Execute Ballot gets document details for the candidate applications
`POST /documents/eligibility-data`

```json
{
  "applicationIds": [101, 102]
}

Response
[
  {
    "applicationId": 101,
    "hasValidHfe": true,
    "documentsComplete": true,
    "extractedFields": {
      "maritalStatus": "MARRIED",
      "citizenship": "SC",
      "householdSize": 2
    }
  },
  {
    "applicationId": 102,
    "hasValidHfe": false,
    "documentsComplete": true,
    "extractedFields": {
      "maritalStatus": "SINGLE",
      "citizenship": "SC",
      "householdSize": 1
    }
  }
]

### Step 7 — Execute Ballot calls EligibilityService to check whether the applications can proceed to BTO ballot
`POST /eligibility/checkEligibility`

```json
{
  "applications": [
    {
      "applicationId": 101,
      "projectId": 11,
      "flatType": "4-Room",
      "documentsComplete": true,
      "extractedFields": {
        "maritalStatus": "MARRIED",
        "citizenship": "SC",
        "householdSize": 2
      }
    },
    {
      "applicationId": 102,
      "projectId": 11,
      "flatType": "4-Room",
      "documentsComplete": true,
      "extractedFields": {
        "maritalStatus": "SINGLE",
        "citizenship": "SC",
        "householdSize": 1
      }
    }
  ]
}

### Step 8 — Execute Ballot gets extra chances from PastApplicationService for the eligible Standard applications
`POST /past-applications/ballot-summary`

```json
{
  "applications": [
    {
      "applicationId": 101,
      "members": [
        {
          "nric": "S1234567A",
          "memberRole": "MAIN_APPLICANT"
        },
        {
          "nric": "S2345678B",
          "memberRole": "CO_APPLICANT"
        }
      ]
    },
    {
      "applicationId": 103,
      "members": [
        {
          "nric": "S4567890D",
          "memberRole": "MAIN_APPLICANT"
        }
      ]
    }
  ]
}

Response
[
  {
    "applicationId": 101,
    "extraChances": 1,
    "hadBookingChanceLast5Years": false,
    "penaltyActive": false
  },
  {
    "applicationId": 103,
    "extraChances": 0,
    "hadBookingChanceLast5Years": true,
    "penaltyActive": false
  }
]



PastApplicationService logic
For family applications:
use MAIN_APPLICANT + CO_APPLICANT as the household key for extra chances

For single applications:
use MAIN_APPLICANT only

Penalty follows both APPLICANT
If penalty is active:
extraChances = 0

Extra chances are based on prior unsuccessful Standard BTO applications for the same household key

### Step 8B — Execute Ballot determines the ballot category and final ballot chances for each eligible application

### Logic
```text
For each eligible application:

1. Read the document facts from Step 6
   - maritalStatus
   - citizenship
   - householdSize
   - member details

2. Read the past application summary from Step 8
   - extraChances
   - hadBookingChanceLast5Years
   - penaltyActive

3. If penaltyActive = true
   - applicantCategory = ST_FAMILY
   - extraChances = 0

4. Else determine the category
   - If the household is a first-timer family:
       - If they satisfy FT(PMC) conditions:
           applicantCategory = FT_PMC
       - Else:
           applicantCategory = FT_FAMILY
   - Else:
       applicantCategory = ST_FAMILY

5. Determine baseChances
   - FT_PMC = 3
   - FT_FAMILY = 2
   - ST_FAMILY = 1

6. Calculate finalChances
   - finalChances = baseChances + extraChances

7. Apply cap
   - FT_PMC max total = 5
   - FT_FAMILY max total = 4
   - ST_FAMILY max total = 1



FT(PMC) if all are satisfied:
- current household is a first-timer family
- and (
    has at least 1 Singapore Citizen child aged 18 and below
    OR married couple aged 40 and below
  )
- and hadBookingChanceLast5Years = false
- and no disqualifying property history based on current eligibility facts

### Step 9 — Execute Ballot calls BallotService to run the ballot for the eligible applications in the bucket
`POST /ballot/run-bucket`

```json
{
  "ballotRunId": 15,
  "exerciseId": 1,
  "projectIds": [11, 12],
  "flatType": "4-Room",
  "availableCount": 280,
  "applications": [
    {
      "applicationId": 101,
      "ballotScheme": "FPPS",
      "applicantCategory": "FT_PMC",
      "finalChances": 4
    },
    {
      "applicationId": 103,
      "ballotScheme": null,
      "applicantCategory": "ST_FAMILY",
      "finalChances": 1
    }
  ]
} 


Response
{
  "status": "COMPLETED",
  "results": [
    {
      "applicationId": 101,
      "queueNo": 1,
      "status": "SUCCESS"
    },
    {
      "applicationId": 103,
      "queueNo": 145,
      "status": "SUCCESS"
    }
  ]
}

## Simplified BallotService logic

1. Receive:
   - `availableCount`
   - `priorityQuota`
   - `applications`

2. Split applications into:
   - priority pool
   - general pool

3. Run weighted random draw for priority pool up to `priorityQuota`

4. Move unsuccessful priority applicants into general pool

5. Run weighted random draw for general pool up to remaining slots

6. Assign queue numbers

7. Return results

### Step 10 — Execute Ballot sends the successful ballot results to FlatSelectionService

### Request
`POST /flat-selection/create`

```json
{
  "ballotRunId": 15,
  "selections": [
    {
      "applicationId": 101,
      "applicantId1": 1,
      "applicantId2": 2,
      "queueNumber": 1,
      "status": "IN_PROGRESS",
      "flatId": null
    },
    {
      "applicationId": 103,
      "applicantId1": 3,
      "applicantId2": null,
      "queueNumber": 145,
      "status": "IN_PROGRESS",
      "flatId": null
    }
  ]
}

Response
{
  "status": "RECEIVED"
}

### Step 11 — Execute Ballot publishes ballot outcome events to RabbitMQ for NotificationService

### Exchange / Queue
`ballot.notifications`

### Message
```json
{
  "ballotRunId": 15,
  "notifications": [
    {
      "applicationId": 101,
      "email": "john@example.com",
      "sms": "+6591234567",
      "queueNumber": 1,
      "status": "SUCCESS"
    },
    {
      "applicationId": 103,
      "email": "mary@example.com",
      "sms": "+6598765432",
      "queueNumber": 145,
      "status": "SUCCESS"
    },
    {
      "applicationId": 104,
      "email": "alex@example.com",
      "sms": "+6591112222",
      "queueNumber": null,
      "status": "UNSUCCESSFUL"
    }
  ]
}

### Step 12 — Execute Ballot updates BallotAuditService with the final ballot run status

### Request
`PUT /ballot-audit/runs/15`

```json
{
  "status": "COMPLETED_WITH_ERRORS"
}

COMPLETED FAIL